# Reading Planetary Computer COGs with async-geotiff

This notebook walks through pixel-level Cloud Optimized GeoTIFF reads with [async-geotiff](https://github.com/developmentseed/async-geotiff). async-geotiff is a Python COG reader with a Rust core, no GDAL dependency, and an async-first API. Key benefits:

1. **No GDAL** — pure pip install. No system libraries, no platform-specific wheels.
2. **Async-first** — `read()` is awaitable. Fire many in parallel with `asyncio.gather`.
3. **Zero-copy** — the Rust core hands NumPy a view into the decoded buffer, no Python-side memory copies.
4. **Type-hinted** — full IDE autocomplete and `mypy` coverage on every API.
5. **Overview-aware** — pick the right resolution explicitly; no surprises from automatic downsampling.

Each cell below demonstrates one of these.

The companion [async-geotiff tutorial](../overview/async-geotiff.md) has the full narrative and the migration notes.

## Install

In [ ]:
%pip install --quiet async-geotiff obstore lonboard planetary-computer pystac-client matplotlib

## Find a Sentinel-2 scene

`modifier=planetary_computer.sign_inplace` signs every asset href as the search returns.

**Expected result:** a printed asset href pointing at a Sentinel-2 B04 COG on Azure Blob.

In [ ]:
import pystac_client
import planetary_computer

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
item = next(catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=[-122.7, 45.5, -122.6, 45.6],
    datetime="2024-07-01/2024-08-01",
    max_items=1,
).items())

asset = item.assets["B04"]
asset.href

## Build an authenticated obstore store

async-geotiff reads bytes through an obstore store. `PlanetaryComputerCredentialProvider` figures out the account, container, and prefix from the asset.

**Expected result:** working `store` object, no output printed.

In [ ]:
from obstore.auth.planetary_computer import PlanetaryComputerCredentialProvider
from obstore.store import AzureStore

provider = PlanetaryComputerCredentialProvider.from_asset(asset, async_=True)
store = AzureStore(credential_provider=provider)

## Open the COG and inspect metadata

Opening reads only the COG header — typically ~16 KB — not the pixel data. The transform tells you the scene's pixel size and ground position; the CRS is its coordinate reference system; `overviews` lists the resolution pyramid finest-to-coarsest.

**Expected result:** a transform, a CRS, a nodata value, and a non-empty overviews list.

In [ ]:
from async_geotiff import GeoTIFF

geotiff = await GeoTIFF.open(asset.href, store=store)
geotiff.transform, geotiff.crs, geotiff.nodata

In [ ]:
[(ov.width, ov.height) for ov in geotiff.overviews]

## Read a window

A *window* is a rectangle in image coordinates. The reader fetches only the COG tiles that intersect it — for a 512×512 window inside a multi-thousand-pixel image, that's a small fraction of the file.

**Expected result:** an array with shape `(1, 512, 512)` and a transform describing the windowed region.

In [ ]:
from async_geotiff import Window

window = Window(col_off=2048, row_off=2048, width=512, height=512)
array = await geotiff.overviews[0].read(window=window)

array.data.shape, array.transform

## Preview the window

**Expected result:** a grayscale plot of the windowed Sentinel-2 B04 reflectance values.

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(array.data[0], cmap="gray")
plt.colorbar()
plt.title("Sentinel-2 B04 window")

## Visualize the full scene with Lonboard

For an interactive map view of the same item, hand it to Lonboard's `RasterLayer.from_stac()`.

**Expected result:** an interactive map of the Sentinel-2 scene that pans and zooms.

In [ ]:
from lonboard import Map
from lonboard.experimental import RasterLayer

Map(RasterLayer.from_stac([item]))

## Read many windows in parallel

Each `read()` is independent. Fire 64 of them at once and let async-geotiff issue range requests in parallel and decode them on the Rust thread pool.

**Expected result:** 64 arrays, each shape `(1, 256, 256)`.

In [ ]:
import asyncio

windows = [
    Window(c, r, 256, 256)
    for c in range(0, 2048, 256)
    for r in range(0, 2048, 256)
]
arrays = await asyncio.gather(
    *[geotiff.overviews[0].read(window=w) for w in windows]
)
len(arrays), arrays[0].data.shape

## You're done

If every cell above produced its expected result, async-geotiff is wired up end-to-end:

- **No GDAL** — pip-installed, no system deps
- **Async-first** — 64 parallel window reads in one `gather`
- **Zero-copy** — array data is a NumPy view onto the decoded buffer
- **Type-hinted** — full autocomplete on `GeoTIFF`, `Window`, `Array`
- **Overview-aware** — picked overview `[0]` for full resolution explicitly

For resampling or reprojection, hand the array to [rasterio](https://rasterio.readthedocs.io/) via an in-memory file. For browser-side rendering of the same data, see the [Lonboard tutorial](../overview/lonboard.md) or the [deck.gl-raster tutorial](../overview/deckgl-raster.md).